## WayFarer: Multi-Agent Holiday Planning System

## Project Overview

WayFarer is a multi-agent holiday planning system developed using Python, LangChain, LangGraph, Model Context Protocol (MCP), Human In The Loop (HITL) middleware and Gemini. It automates holiday planning through a supervisor-subagent architecture, coordinating multiple specialized agents to provide personalized travel options, generate itineraries and emails them to the user with a human approval.

## Architecture and Methodology

<b>1) Supervisor Agent:</b> Coordinates the overall workflow, manages user requests, and delegates tasks to specialized subagents.

<b>2) Specialized Subagents:</b> Implements three subagents responsible for:
    
    a) Flight Agent: Retrieves suitable flight options.
    
    b) Accommodation Agent: Finds accommodation options.
    
    c) Local Experience Agent: Retrieves local attractions and experiences.

<b>3) Conversational Memory:</b> Maintains conversation context to handle follow-up requests and refine travel plans.

<b>4) MCP Integration:</b> Connects the subagents to the BreckenWander MCP server through Streamable HTTP to retrieve relevant travel information.

<b>5) Itinerary Generation:</b> Combines the retrieved options to generate holiday itineraries with associated prices and booking links.

<b> 6) Human In The Loop (HITL) Middleware:</b> Emails the generated itinerary to the user's email id with a human approval.

## Key Feature

The system uses a supervisor-subagent architecture with specialized agents and conversational memory to coordinate travel planning, retrieve relevant options, and generate comprehensive holiday itineraries.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "breckenwander": {
            "transport": "streamable_http",     
            "url": "https://mcp.breckenwander.com/mcp"
        }
    }
)

tools = await client.get_tools()
print([t.name for t in tools])

['search_flights', 'search_hotels', 'search_experiences', 'build_trip']


In [4]:
flight_tools = [t for t in tools if t.name == "search_flights"]
hotel_tools = [t for t in tools if t.name == "search_hotels"]
attraction_tools = [t for t in tools if t.name == "search_experiences"]

MODEL = "google_genai:gemini-3.5-flash-lite"

In [5]:
from langchain.agents import AgentState

class HolidayState(AgentState):      
    origin: str
    destination: str
    start_date: str
    end_date: str
    travelers: str

In [6]:
from langchain.agents import create_agent

# Flight agent
flight_agent = create_agent(
    model=MODEL,
    tools=flight_tools,
    system_prompt='''
You are a flight agent. Search for round-trip flights between the given origin
and destination. You are not allowed to ask follow-up questions. You must find
the best flight options based on:
- Price (lowest)
- Duration (shortest)
- Dates given
Convert city names to airport codes yourself. You will be given the origin,
destination, dates and number of travelers. You may need to make multiple
searches to identify the best options. Once you have found the best options,
return a shortlist of 2-3 options with price (USD), duration and booking link.
'''
)

In [7]:
# Hotel agent
hotel_agent = create_agent(
    model=MODEL,
    tools=hotel_tools,
    system_prompt='''
You are a hotel agent. Search for hotels in the given destination for the given
dates and number of travelers. You are not allowed to ask follow-up questions.
Choose the best options based on:
- All-in price (lowest)
- Quality / rating
Return a shortlist of 2-3 hotels with all-in total price (USD), a one-line
reason for each pick and the booking link.
'''
)

In [8]:
# Attractions agent
attraction_agent = create_agent(
    model=MODEL,
    tools=attraction_tools,
    system_prompt='''
You are a local experiences agent. Search for tours, activities and tourist
experiences in the given destination. You are not allowed to ask follow-up
questions. Return 4-6 varied suggestions (for example culture, food, outdoors)
with price (USD), a one-line description and the booking link.
'''
)

In [9]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

REQUIRED = ["origin", "destination", "start_date", "end_date", "travelers"]

def missing_state(runtime: ToolRuntime) -> list:
    '''Return the state keys that haven't been filled in yet'''
    return [k for k in REQUIRED if not runtime.state.get(k)]

@tool
async def find_flights(runtime: ToolRuntime) -> str:
    '''Flight agent searches round-trip flights'''
    missing = missing_state(runtime)
    if missing:
        return (f"State is not ready (missing: {missing}). Call update_state first, "
                "wait for it to finish, then call this tool again.")
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    start_date = runtime.state["start_date"]
    end_date = runtime.state["end_date"]
    travelers = runtime.state["travelers"]
    query = (f"Find round-trip flights from {origin} to {destination}, "
             f"departing {start_date} and returning {end_date}, for {travelers} travelers")
    response = await flight_agent.ainvoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].text

@tool
async def find_hotels(runtime: ToolRuntime) -> str:
    '''Hotel agent searches accommodation'''
    missing = missing_state(runtime)
    if missing:
        return (f"State is not ready (missing: {missing}). Call update_state first, "
                "wait for it to finish, then call this tool again.")
    destination = runtime.state["destination"]
    start_date = runtime.state["start_date"]
    end_date = runtime.state["end_date"]
    travelers = runtime.state["travelers"]
    query = (f"Find hotels in {destination} from {start_date} to {end_date} "
             f"for {travelers} travelers")
    response = await hotel_agent.ainvoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].text

@tool
async def find_attractions(runtime: ToolRuntime) -> str:
    '''Attractions agent suggests local tourist spots and experiences'''
    missing = missing_state(runtime)
    if missing:
        return (f"State is not ready (missing: {missing}). Call update_state first, "
                "wait for it to finish, then call this tool again.")
    destination = runtime.state["destination"]
    start_date = runtime.state["start_date"]
    end_date = runtime.state["end_date"]
    query = (f"Find tourist experiences and activities in {destination} "
             f"between {start_date} and {end_date}")
    response = await attraction_agent.ainvoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].text

@tool
def update_state(origin: str, destination: str, start_date: str, end_date: str,
                 travelers: str, runtime: ToolRuntime) -> str:
    '''Update the state when you know all 5 values: origin, destination,
    start_date, end_date, travelers'''
    return Command(update={
        "origin": origin,
        "destination": destination,
        "start_date": start_date,
        "end_date": end_date,
        "travelers": travelers,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]
    })

In [10]:
from langgraph.checkpoint.memory import InMemorySaver

coordinator = create_agent(
    model=MODEL,
    tools=[find_flights, find_hotels, find_attractions, update_state],
    state_schema=HolidayState,
    checkpointer=InMemorySaver(),
    system_prompt='''
You are a holiday planning coordinator. Follow these steps:
1. Make sure you know all 5 key details: origin, destination, start_date,
   end_date and number of travelers. If any is missing, ask the user for it in
   one short message. Never guess dates. Use the YYYY-MM-DD format.
2. Once you know all 5, call update_state.
3. Then call find_flights, find_hotels and find_attractions.
4. Compile the results into one clear holiday plan: flight shortlist, hotel
   shortlist and things to do, with booking links. Mention that prices are in
   USD and may change until checkout.
'''
)

In [11]:
config = {"configurable": {"thread_id": "1"}}

response = await coordinator.ainvoke(
    {"messages": [HumanMessage(content="I'm flying from Bangalore and I'd like a holiday in Seattle from 2027-03-14 to 2027-03-21 for 3 travelers")]},
    config
)

Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring


In [12]:
print(response["messages"][-1].text)

Here is your holiday plan for **Seattle** from **March 14, 2027 to March 21, 2027** for **3 travelers**, departing from **Bangalore**. 

---

### ✈️ Flight Shortlist (Round-Trip)
Prices below are per traveler:
1. **Lufthansa** (1 stop via MUC) — $1,375 / traveler *(Package-only fare)* | [Book Lufthansa](https://mcp.breckenwander.com/go?q=e9725c6a-3ca5-4563-945e-087132e32654&sig=ae1100fd312324eab4b1d4b48575338e1818b1035c189e23d4bda6153c28ff9f&t=flight&s=other&v=text)
2. **Qatar Airways** (1 stop via DOH) — $1,426 / traveler | [Book Qatar Airways](https://mcp.breckenwander.com/go?q=75d4a5cb-a189-462e-a968-66d697400676&sig=faa1a0a679ea0cfe1c08b3ba7dc641db99ce52f791d948316f5996e96b4666cd&t=flight&s=other&v=text)
3. **Lufthansa** (1 stop via FRA) — $1,445 / traveler | [Book Lufthansa](https://mcp.breckenwander.com/go?q=fb123d8a-4bcf-4f6a-8993-8ffdfc5cc0e0&sig=13606fa693908778ee087696b9832db1136715b09efc1476023dc6d04d5f8cc6&t=flight&s=other&v=text)

---

### 🏨 Hotel Shortlist (7 Nights)
Pric

In [14]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
import os, smtplib, uuid
from email.message import EmailMessage

to_email = "abc@gmail.com"      

@tool
def send_email(to: str, body: str) -> str:
    '''Send an email to the given address with the given subject & body'''
    msg = EmailMessage()
    msg["From"] = os.getenv("EMAIL_ADDRESS")
    msg["To"] = to
    msg["Subject"] = "Your Holiday Itinerary"
    msg.set_content(body)
    with smtplib.SMTP_SSL("smtp.mail.yahoo.com", 465) as server:
        server.login(os.getenv("EMAIL_ADDRESS"), os.getenv("EMAIL_APP_PASSWORD"))
        server.send_message(msg)
    return f"Email sent to {to}"

email_agent = create_agent(
    model=MODEL,
    tools=[send_email],
    system_prompt="You send emails. Always call the send_email tool immediately with the full text you are given as the body. Never ask questions and never reply with a draft.",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware[AgentState, None](
            interrupt_on={"send_email": True},   
            description_prefix="Tool execution requires approval",
        ),
    ],
)

config = {"configurable": {"thread_id": str(uuid.uuid4())}}    # fresh thread every run

email_response = email_agent.invoke(
    {"messages": [HumanMessage(content=f"Send this holiday itinerary by email to {to_email}:\n\n" + response["messages"][-1].text)]},
    config=config
)

if input("Send this email? (yes/no): ").strip().lower() == "yes":
    decision = {"type": "approve"}
else:
    decision = {"type": "reject", "message": "No, please do not send this email."}

email_response = email_agent.invoke(
    Command[tuple[()]](resume={"decisions": [decision]}),
    config=config      
)

print(email_response["messages"][-1].text)

Send this email? (yes/no):  yes
